In [0]:
%run ./00_config

In [0]:
from pyspark.sql.functions import sum, count, round

# 1. Read cleansed data from Silver
df_silver = spark.read.format("delta").load(silver_path)

# 2. Compute Gold business aggregations
df_gold = (
    df_silver
    .groupBy("customer_id")
    .agg(
        count("order_id").alias("total_orders"),
        round(sum("transaction_amount"), 2).alias("total_spend")
    )
)

# 3. Write reporting-ready summary to Gold
df_gold.write.format("delta").mode("overwrite").save(gold_path)
print("Gold aggregation written successfully!")

In [0]:
df_gold_check = spark.read.format("delta").load(gold_path)
print(f"Gold layer verified! Row count: {df_gold_check.count()}")
display(df_gold_check)